# Transfer Learning v3 — Modern CNN Backbones

Replaces the dated VGG / ResNet18 backbones with architectures designed in 2021-2022
that significantly outperform VGG on ImageNet at equal or smaller model size.

| Backbone | Year | Top-1 | Params | Notes |
|---|---|---|---|---|
| **ConvNeXt-Tiny** | 2022 | 82.1% | 28 M | Best pure-CNN; matches ViT with standard training recipe. |
| **EfficientNetV2-S** | 2021 | 83.9% | 21 M | Improved training-aware NAS over original EfficientNet. |
| **ResNet50** | 2015 | 80.9% | 25 M | Stronger baseline than ResNet18 (2× depth). |

Everything else is identical to v2:
- 224 × 224 input, RandAugment + RandomErasing
- Asymmetric Loss (γ⁻=4, γ⁺=1)
- Stratified multi-label split
- Two-stage fine-tune: freeze → full unfreeze with discriminative LRs
- Per-class threshold tuning on val

## 1. Imports & Setup

In [ ]:
import sys
import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from pathlib import Path
from collections import defaultdict
from torchvision import models as tv_models
from torchvision import transforms
from torchvision.transforms import RandAugment, RandomErasing
from torch.utils.data import DataLoader, Subset

sys.path.append("../")
from eval import LABEL_ORDER
from utils import (
    set_seed, load_dataset,
    plot_per_class_examples, plot_multilabel_examples,
    run_baselines, print_model_info,
    save_checkpoint,
    plot_multi_arch_histories,
    collect_test_predictions, categorize_predictions,
    show_prediction_examples, plot_per_class_metrics,
    plot_confusion_matrices, plot_prediction_heatmap,
    show_saliency_examples, compute_multilabel_metrics,
    NUM_LABELS, METRIC_KEYS, NORM_MEAN, NORM_STD, TransformSubset,
)

SEED = 42
set_seed(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 2. Config

In [ ]:
BASE_DATA_DIR    = "../data/aggregated"
IMAGE_SIZE       = 224
BATCH_SIZE       = 32
SPLIT            = [0.7, 0.15, 0.15]
USE_SUBSET       = False
SUBSET_FRACTION  = 0.1
CHECKPOINT_DIR   = Path("../checkpoints")

# Two-stage training
FREEZE_EPOCHS    = 3
UNFREEZE_EPOCHS  = 17
MAX_EPOCHS       = FREEZE_EPOCHS + UNFREEZE_EPOCHS
LR_HEAD          = 1e-3
LR_BACKBONE      = 1e-4
WEIGHT_DECAY     = 1e-4
GRAD_CLIP        = 1.0
THRESHOLD        = 0.5
THRESHOLD_GRID   = np.arange(0.05, 0.96, 0.02)

print(f"Labels ({NUM_LABELS}): {LABEL_ORDER}")
print(f"Total epochs: {MAX_EPOCHS}  (freeze={FREEZE_EPOCHS}, unfreeze={UNFREEZE_EPOCHS})")

## 3. Asymmetric Loss

In [ ]:
class AsymmetricLoss(nn.Module):
    """Asymmetric Loss (Ben-Baruch et al. 2020)."""

    def __init__(self, gamma_neg: float = 4, gamma_pos: float = 1,
                 clip: float = 0.05, eps: float = 1e-8):
        super().__init__()
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.eps = eps

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        probs_pos = torch.sigmoid(logits)
        probs_neg = 1 - probs_pos
        if self.clip > 0:
            probs_neg = (probs_neg + self.clip).clamp(max=1.0)
        loss_pos = targets       * torch.log(probs_pos.clamp(min=self.eps))
        loss_neg = (1 - targets) * torch.log(probs_neg.clamp(min=self.eps))
        loss = loss_pos + loss_neg
        pt    = probs_pos * targets + probs_neg * (1 - targets)
        gamma = self.gamma_pos * targets + self.gamma_neg * (1 - targets)
        return -(loss * (1 - pt).pow(gamma)).mean()

criterion = AsymmetricLoss(gamma_neg=4, gamma_pos=1, clip=0.05)

## 4. Data — Stratified Split + RandAugment

In [ ]:
def stratified_split_multilabel(dataset, split, seed=42):
    combo_to_indices = defaultdict(list)
    for i in range(len(dataset)):
        _, target = dataset[i]
        combo_to_indices[tuple(target.int().tolist())].append(i)
    rng = np.random.default_rng(seed)
    train_idx, val_idx, test_idx = [], [], []
    for indices in combo_to_indices.values():
        indices = np.array(indices)
        rng.shuffle(indices)
        n       = len(indices)
        n_val   = max(1 if n >= 3 else 0, round(split[1] * n))
        n_test  = max(1 if n >= 3 else 0, round(split[2] * n))
        n_train = max(0, n - n_val - n_test)
        train_idx.extend(indices[:n_train].tolist())
        val_idx.extend(indices[n_train : n_train + n_val].tolist())
        test_idx.extend(indices[n_train + n_val :].tolist())
    return Subset(dataset, train_idx), Subset(dataset, val_idx), Subset(dataset, test_idx)

In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(IMAGE_SIZE, scale=(0.8, 1.0)),
    transforms.RandomHorizontalFlip(),
    RandAugment(num_ops=2, magnitude=9),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
    RandomErasing(p=0.25, scale=(0.02, 0.1), value=0.0),
])

eval_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=NORM_MEAN, std=NORM_STD),
])

full_dataset = load_dataset(BASE_DATA_DIR)
train_raw, val_raw, test_raw = stratified_split_multilabel(full_dataset, SPLIT, SEED)

if USE_SUBSET:
    from utils import subsample_subset
    train_raw = subsample_subset(train_raw, SUBSET_FRACTION, SEED)
    val_raw   = subsample_subset(val_raw,   SUBSET_FRACTION, SEED + 1)
    test_raw  = subsample_subset(test_raw,  SUBSET_FRACTION, SEED + 2)

train_ds = TransformSubset(train_raw, transform=train_transform)
val_ds   = TransformSubset(val_raw,   transform=eval_transform)
test_ds  = TransformSubset(test_raw,  transform=eval_transform)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=4, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=4, pin_memory=True)

print(f"Total: {len(full_dataset)}  |  Train: {len(train_raw)}  Val: {len(val_raw)}  Test: {len(test_raw)}")

## 5. Baselines

In [ ]:
run_baselines(train_loader, val_loader, test_loader, NUM_LABELS, DEVICE)

## 6. Model — Modern CNN Backbones

All models are wrapped in `TransferModel` for two-stage training.

In [ ]:
class TransferModel(nn.Module):
    def __init__(self, backbone: nn.Module, head: nn.Module):
        super().__init__()
        self.backbone = backbone
        self.head     = head

    def forward(self, x):
        return self.head(self.backbone(x))

    def set_backbone_trainable(self, flag: bool):
        for p in self.backbone.parameters():
            p.requires_grad = flag


def build_model(arch: str, num_labels: int) -> TransferModel:
    if arch == "convnext_tiny":
        m        = tv_models.convnext_tiny(weights=tv_models.ConvNeXt_Tiny_Weights.IMAGENET1K_V1)
        # ConvNeXt: features → avgpool → flatten
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = nn.Linear(768, num_labels)

    elif arch == "efficientnet_v2_s":
        m        = tv_models.efficientnet_v2_s(weights=tv_models.EfficientNet_V2_S_Weights.IMAGENET1K_V1)
        backbone = nn.Sequential(m.features, m.avgpool, nn.Flatten())
        head     = nn.Sequential(nn.Dropout(0.2), nn.Linear(1280, num_labels))

    elif arch == "resnet50":
        m        = tv_models.resnet50(weights=tv_models.ResNet50_Weights.IMAGENET1K_V2)
        backbone = nn.Sequential(*list(m.children())[:-1], nn.Flatten())
        head     = nn.Linear(2048, num_labels)

    else:
        raise ValueError(f"Unknown arch: {arch}")

    model = TransferModel(backbone, head)
    # Initialise new head weights
    for layer in model.head.modules():
        if isinstance(layer, nn.Linear):
            nn.init.normal_(layer.weight, 0, 0.01)
            nn.init.zeros_(layer.bias)
    return model


ARCHS = ["convnext_tiny", "efficientnet_v2_s", "resnet50"]

print(f"{'Arch':<22} {'Total params':>14}  {'Size (MB)':>10}")
print("-" * 52)
for arch in ARCHS:
    m     = build_model(arch, NUM_LABELS)
    total = sum(p.numel() for p in m.parameters())
    print(f"{arch:<22} {total:>14,}  {total*4/1024/1024:>10.2f}")

## 7. Training — Two-Stage Fine-Tune

In [ ]:
def run_epoch(model, loader, optimizer=None, train=True):
    model.train(train)
    all_logits, all_probs, all_preds, all_labels = [], [], [], []
    total_loss = 0.0
    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.set_grad_enabled(train):
            logits = model(images)
            loss   = criterion(logits, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(
                    [p for p in model.parameters() if p.requires_grad], GRAD_CLIP
                )
                optimizer.step()
        with torch.no_grad():
            probs = torch.sigmoid(logits)
            preds = (probs >= THRESHOLD).float()
        total_loss  += loss.item() * images.size(0)
        all_logits.append(logits.detach().cpu())
        all_probs.append(probs.detach().cpu())
        all_preds.append(preds.detach().cpu())
        all_labels.append(labels.cpu())
    n = sum(l.size(0) for l in all_labels)
    metrics = compute_multilabel_metrics(
        torch.cat(all_labels), torch.cat(all_preds),
        probs=torch.cat(all_probs), logits=torch.cat(all_logits),
    )
    metrics["loss"] = total_loss / n
    return metrics


def train_two_stage(arch):
    model = build_model(arch, NUM_LABELS).to(DEVICE)
    model.set_backbone_trainable(False)
    optimizer = optim.AdamW(
        [{"params": list(model.head.parameters()), "lr": LR_HEAD}],
        weight_decay=WEIGHT_DECAY,
    )
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)
    best_val_f1, best_state = -1.0, None
    history = {"train": {k: [] for k in METRIC_KEYS}, "val": {k: [] for k in METRIC_KEYS}}

    print(f"[Stage A] backbone frozen for {FREEZE_EPOCHS} warmup epochs")
    for epoch in range(1, MAX_EPOCHS + 1):
        if epoch == FREEZE_EPOCHS + 1:
            model.set_backbone_trainable(True)
            optimizer = optim.AdamW(
                [
                    {"params": list(model.head.parameters()),     "lr": LR_HEAD},
                    {"params": list(model.backbone.parameters()), "lr": LR_BACKBONE},
                ],
                weight_decay=WEIGHT_DECAY,
            )
            scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=UNFREEZE_EPOCHS)
            print(f"[Stage B] backbone unfrozen — full fine-tune for {UNFREEZE_EPOCHS} epochs")

        t0      = time.time()
        train_m = run_epoch(model, train_loader, optimizer, train=True)
        val_m   = run_epoch(model, val_loader, train=False)
        scheduler.step()

        for k in METRIC_KEYS:
            history["train"][k].append(train_m[k])
            history["val"][k].append(val_m[k])

        stage = "A" if epoch <= FREEZE_EPOCHS else "B"
        print(f"[{stage}] {epoch:02d}/{MAX_EPOCHS} | {time.time()-t0:4.1f}s | "
              f"train f1={train_m['f1_micro']:.4f} | val f1={val_m['f1_micro']:.4f}")
        if val_m["f1_micro"] > best_val_f1:
            best_val_f1 = val_m["f1_micro"]
            best_state  = copy.deepcopy(model.state_dict())
            print(f"       -> new best {best_val_f1:.4f}")

    model.load_state_dict(best_state)
    model.eval()
    return model, best_val_f1, history

In [ ]:
all_histories = {}
best_models   = {}
best_val_f1s  = {}

for arch in ARCHS:
    print(f"\n{'='*60}\n  {arch}\n{'='*60}")
    m, best_f1, history = train_two_stage(arch)
    save_checkpoint(m.state_dict(), CHECKPOINT_DIR / f"tl_v3_{arch}.pth")
    all_histories[arch] = {k: {"train": history["train"][k], "val": history["val"][k]}
                           for k in METRIC_KEYS}
    best_models[arch]   = m
    best_val_f1s[arch]  = best_f1

print("\n=== Val F1 Summary ===")
for arch in ARCHS:
    print(f"  {arch:<22} {best_val_f1s[arch]:.4f}")

plot_multi_arch_histories(all_histories, experiment_name="Transfer Learning v3")

## 8. Per-Class Threshold Tuning

In [ ]:
def tune_thresholds(model, val_loader, grid):
    all_probs, all_labels = [], []
    model.eval()
    with torch.no_grad():
        for images, labels in val_loader:
            probs = torch.sigmoid(model(images.to(DEVICE)))
            all_probs.append(probs.cpu())
            all_labels.append(labels.cpu())
    probs  = torch.cat(all_probs)
    labels = torch.cat(all_labels)
    thresholds = []
    for c in range(labels.shape[1]):
        best_t, best_f1 = 0.5, -1.0
        for t in grid:
            preds = (probs[:, c] >= t).float()
            tp = ((preds == 1) & (labels[:, c] == 1)).sum().float()
            fp = ((preds == 1) & (labels[:, c] == 0)).sum().float()
            fn = ((preds == 0) & (labels[:, c] == 1)).sum().float()
            f1 = (2 * tp / (2 * tp + fp + fn + 1e-8)).item()
            if f1 > best_f1:
                best_f1, best_t = f1, t
        thresholds.append(float(best_t))
    return thresholds, probs, labels


per_class_thresholds = {}
print("Tuning per-class thresholds on val set...\n")
for arch, m_eval in best_models.items():
    thresholds, val_probs, val_labels = tune_thresholds(m_eval, val_loader, THRESHOLD_GRID)
    per_class_thresholds[arch] = thresholds
    thr_t     = torch.tensor(thresholds).unsqueeze(0)
    val_preds = (val_probs >= thr_t).float()
    m         = compute_multilabel_metrics(val_labels, val_preds, probs=val_probs)
    print(f"{arch:<22}  val F1@tuned: {m['f1_micro']:.4f}")
    for i, cls in enumerate(LABEL_ORDER):
        print(f"  {cls:<12}  thr={thresholds[i]:.2f}")
    print()

## 9. Test Evaluation

In [ ]:
print(f"\n{'Arch':<22} {'F1@0.5':>8} {'F1@tuned':>10} {'Exact':>7} {'IoU':>7}")
print("-" * 62)
for arch, m_eval in best_models.items():
    thr_t  = torch.tensor(per_class_thresholds[arch]).unsqueeze(0)
    all_l, all_pr, all_lg = [], [], []
    with torch.no_grad():
        for images, labels in test_loader:
            logits = m_eval(images.to(DEVICE))
            all_l.append(labels.cpu())
            all_pr.append(torch.sigmoid(logits).cpu())
            all_lg.append(logits.cpu())
    labels_t = torch.cat(all_l)
    probs_t  = torch.cat(all_pr)
    logits_t = torch.cat(all_lg)
    m_def   = compute_multilabel_metrics(labels_t, (probs_t >= 0.5).float(),
                                         probs=probs_t, logits=logits_t)
    m_tuned = compute_multilabel_metrics(labels_t, (probs_t >= thr_t).float(),
                                         probs=probs_t, logits=logits_t)
    print(f"{arch:<22} {m_def['f1_micro']:>8.4f} {m_tuned['f1_micro']:>10.4f} "
          f"{m_tuned['exact_match']:>7.4f} {m_tuned['mean_iou']:>7.4f}")

## 10. Analyze Predictions (Best Architecture)

In [ ]:
BEST_ARCH  = max(best_val_f1s, key=best_val_f1s.get)
model      = best_models[BEST_ARCH]
thr_tensor = torch.tensor(per_class_thresholds[BEST_ARCH]).unsqueeze(0)
print(f"Best arch: {BEST_ARCH}  (val F1={best_val_f1s[BEST_ARCH]:.4f})")

images_t, labels_t, _, probs_t = collect_test_predictions(model, test_loader, DEVICE)
preds_t = (probs_t >= thr_tensor).float()
correct_idx, partial_idx, incorrect_idx = categorize_predictions(labels_t, preds_t)

show_prediction_examples(correct_idx,   images_t, labels_t, preds_t, "Fully Correct",     n=4)
show_prediction_examples(partial_idx,   images_t, labels_t, preds_t, "Partially Correct", n=4)
show_prediction_examples(incorrect_idx, images_t, labels_t, preds_t, "Fully Incorrect",   n=4)

In [ ]:
plot_per_class_metrics(labels_t, preds_t)
plot_confusion_matrices(labels_t, preds_t)
plot_prediction_heatmap(labels_t, preds_t)

## 11. Saliency Maps

In [ ]:
show_saliency_examples(correct_idx,   images_t, labels_t, preds_t, model, "Fully Correct",     n=5, device=DEVICE)
show_saliency_examples(partial_idx,   images_t, labels_t, preds_t, model, "Partially Correct", n=5, device=DEVICE)
show_saliency_examples(incorrect_idx, images_t, labels_t, preds_t, model, "Fully Incorrect",   n=5, device=DEVICE)